In [1]:
import pandas as pd

df = pd.read_csv("/content/soil_small_pre.csv")

df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,141,64,137,26.19,92.70,6.75,1699.74,Banana
1,148,77,149,29.51,74.47,7.22,2276.49,Banana
2,103,67,100,27.80,80.64,6.26,1577.72,Banana
3,138,60,145,28.80,79.38,6.22,2424.47,Banana
4,102,49,104,29.79,78.97,6.32,2078.95,Banana


In [2]:
df["label"].value_counts()

,count
label,
Coffee,40
Banana,15
Ginger,15
Pepper,15
Tapioca,15


In [3]:
X = df.drop("label", axis=1)
y = df["label"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [8]:
from sklearn.metrics import accuracy_score
import pandas as pd

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
),
    "SVM": SVC(),
    "Gradient Boosting": GradientBoostingClassifier()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results.append((name, acc))

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"])
results_df.sort_values(by="Accuracy", ascending=False)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy
3,Random Forest,0.95
2,Decision Tree,0.90
4,SVM,0.85
0,Logistic Regression,0.85
5,Gradient Boosting,0.85
1,KNN,0.80


In [9]:
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 1.0
Test Accuracy: 0.85


In [10]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

print("DT Train:", accuracy_score(y_train, dt.predict(X_train)))
print("DT Test:", accuracy_score(y_test, dt.predict(X_test)))

DT Train: 1.0
DT Test: 0.9


In [11]:
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(n_estimators=150, random_state=42)

scores = cross_val_score(rf, X, y, cv=5)

print("Cross-val scores:", scores)
print("Mean accuracy:", scores.mean())

Cross-val scores: [0.9  0.85 1.   0.95 0.9 ]
Mean accuracy: 0.9200000000000002


In [15]:
#final model on full data set
final_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=2,
    random_state=42
)

final_model.fit(X, y)

RandomForestClassifier(max_depth=6, min_samples_leaf=2, n_estimators=500,
                       random_state=42)

In [16]:
#testing probability
sample = X.iloc[[0]]
probs = final_model.predict_proba(sample)

classes = final_model.classes_

prob_df = pd.DataFrame(probs, columns=classes)
prob_df


,Banana,Coffee,Ginger,Pepper,Tapioca
0,0.885397,0.001667,0.085174,0.015295,0.012467


In [ ]:
#saving pkl
import pickle

with open("soil_without_smote.pkl", "wb") as f:
    pickle.dump(final_model, f)